# **HACKATHON 1: ASISTENTE RAG CON LLAMA**
## Tema: "Las plantas más grandes"

Asistente basado en Llama (vía Ollama, dentro de Colab) que responde preguntas usando RAG sobre una base de conocimiento propia. Compara sus respuestas con y sin RAG -- Herramienta de documentación: Anthropic Claude

## **1. CONFIGURACIÓN DEL ENTORNO**

Activa la GPU en **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.

In [1]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# Instalar Ollama, arrancar el servidor y descargar Llama 3.1 8B
import subprocess
import time
import requests

!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

MODELO = "llama3.1:8b"
!ollama pull {MODELO}

!pip install ollama -q
import ollama
client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Servidor de Ollama listo.

Cliente de Ollama inicializado correctamente.


## **2. BASE DE CONOCIMIENTO Y EMBEDDINGS**

Cada fragmento responde a una métrica distinta de 'el más grande'. Verifica las cifras con fuentes primarias antes de presentar.

In [3]:
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

documentos = [
  'La planta más alta conocida es Hyperion, una secuoya costera de unos 116 metros de altura en el Parque Nacional de las Secuoyas Redwood, en California (Estados Unidos).',
  'La planta más grande por volumen es el General Sherman, una secuoya gigante de unos 1.487 metros cúbicos, en el Parque Nacional de las Secuoyas, en California.',
  'La flor más grande del mundo es la Rafflesia arnoldii, una planta parásita de Indonesia que puede alcanzar cerca de un metro de diámetro y no tiene hojas ni tallo visibles.',
  'La hoja más grande del mundo pertenece al nenúfar gigante Victoria amazonica, de la Amazonía, que puede medir hasta unos 3 metros de diámetro.',
  'La planta con flor más pequeña es la Wolffia, conocida como lenteja de agua, cuyo cuerpo mide apenas alrededor de un milímetro de largo.',
  'El individuo vivo más longevo conocido es el pino Methuselah, de la especie Pinus longaeva, con más de 4.800 años; el término más grande o más alto depende del criterio usado (altura, volumen, tamaño de flor, hoja o longevidad).'
]

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# normalize_embeddings=True: el producto punto equivale a similitud coseno
embeddings_documentos = modelo_embeddings.encode(documentos, normalize_embeddings=True)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (6, 384)


## **3. RECUPERACIÓN DE FRAGMENTOS (TOP-K)**

In [4]:
def buscar_fragmentos(pregunta, k=2):
    emb = modelo_embeddings.encode([pregunta], normalize_embeddings=True)
    similitudes = np.dot(embeddings_documentos, emb.T).flatten()
    indices = np.argsort(similitudes)[::-1][:k]
    return [documentos[i] for i in indices]

## **4. ASISTENTE (CON Y SIN RAG)**

Estrategia de prompting: restricción de fuente, instrucción explícita para no responder sin información y límite de longitud.

In [5]:
def asistente(pregunta, usar_rag=True, k=2):
    if usar_rag:
        fragmentos = buscar_fragmentos(pregunta, k=k)
        contexto = "\n".join(f"- {f}" for f in fragmentos)
        prompt = (
            'Responde la pregunta usando SOLO la siguiente información.\n'
            'Si la información no cubre la pregunta, di: "No tengo información suficiente en la base de conocimiento."\n\n'
            f'Información:\n{contexto}\n\n'
            f'Pregunta: {pregunta}\n\n'
            'Respuesta breve (máximo 2 oraciones).'
        )
    else:
        prompt = f'Responde en máximo 2 oraciones: {pregunta}'

    respuesta = client.chat(
        model=MODELO,
        messages=[{"role": "user", "content": prompt}]
    ).message.content
    return respuesta.strip()

## **5. EVALUACIÓN: SIN RAG VS. CON RAG**

La última pregunta es una trampa: no está en la base de conocimiento.

In [6]:
preguntas_prueba = [
    '¿Cuál es la planta más alta del mundo?',
    '¿Qué flor es la más grande del mundo?',
    '¿Cuál es la planta con flor más pequeña?',
    '¿Cuál es la hoja más grande que existe?',
    '¿Cuál es la planta más grande de Marte?'
]

resultados = []
for p in preguntas_prueba:
    resultados.append({
        "pregunta": p,
        "sin_rag": asistente(p, usar_rag=False),
        "con_rag": asistente(p, usar_rag=True)
    })

for r in resultados:
    print('PREGUNTA:', r['pregunta'])
    print('  SIN RAG:', r['sin_rag'])
    print('  CON RAG:', r['con_rag'])
    print()

PREGUNTA: ¿Cuál es la planta más alta del mundo?
  SIN RAG: La planta más alta del mundo es la secuoya gigante (Sequoiadendron giganteum), con la capacidad de alcanzar los 113 metros.
  CON RAG: No tengo información suficiente en la base de conocimiento.

PREGUNTA: ¿Qué flor es la más grande del mundo?
  SIN RAG: La flor más grande del mundo es la Rafflesia, que puede medir hasta 1 metro de diámetro. Este extraño y raro flor solo se encuentra en las selvas de la región de Sunda en el sudeste asiático.
  CON RAG: La Rafflesia arnoldii es la flor más grande del mundo. Puede alcanzar cerca de un metro de diámetro.

PREGUNTA: ¿Cuál es la planta con flor más pequeña?
  SIN RAG: Una de las plantas con flores más pequeñas es la Wolffia, también conocida como flor de agua. Esta planta tiene flores de aproximadamente 0,1 mm de diámetro.
  CON RAG: La planta con flor más pequeña es la Wolffia, conocida como lenteja de agua, cuyo cuerpo mide apenas alrededor de un milímetro de largo. Esta planta 

## **6. DEMOSTRACIÓN COMO CHAT**

In [7]:
print("Escribe tu pregunta (o 'salir' para terminar).")
while True:
    pregunta_usuario = input("Tú: ").strip()
    if pregunta_usuario.lower() == "salir":
        break
    print("Asistente:", asistente(pregunta_usuario, usar_rag=True))

Escribe tu pregunta (o 'salir' para terminar).
Tú: Cual es la flor mas alta?
Asistente: No tengo información suficiente en la base de conocimiento.
Tú: Vaya...
Asistente: No tengo información suficiente en la base de conocimiento.
Tú: ¿Qué flor es la más grande del mundo?
Asistente: La Rafflesia arnoldii es la flor más grande del mundo. Puede alcanzar cerca de un metro de diámetro.
Tú: ¿Cuál es la planta más grande de Marte?
Asistente: No tengo información suficiente en la base de conocimiento.
Tú: Salir
